In [1]:
import sys
import torch
import torch.nn as nn
from torch.nn import functional as F
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime

from torch.profiler import profile, record_function, ProfilerActivity
from torch.cuda.amp import GradScaler, autocast

In [2]:
import wandb
wandb.login()

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
wandb: Currently logged in as: merhawi (ajax_m). Use `wandb login --relogin` to force relogin


True

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
torch.manual_seed(1337)

cuda


In [4]:
block_size = 1024
batch_size = 32*2
max_iters = 1001
learning_rate = 3e-4
eval_iters = 200
eval_interval = 200
n_embed = 384
n_layer = 4
n_head = 4
dropout = 0.2

# wandb tracking initialization

wandb.init(project = 'nano-gpt-tracking-test',
      config={
            "block_size" : 1024,
            "batch_size" :32*2,
            "max_iters" :1001,
            "eval_iterval" : 200,
            "lr" : 3e-4,
            "eval_iters" : 200,
            "n_emb" : 384,
            "n_layer" : 4,
            "n_head" :4,
            "dropout" : 0.2,
            #"dtype" : 'bfloat16' didnt work yet need some debugging,
}
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
chars = ""
data_used = '../data/input.txt'
with open(data_used, 'r', encoding='utf-8')as f:
    text = f.read()
    chars = sorted(set(text))

vocab_size = len(chars)

size_mb = sys.getsizeof(text) / (1024 * 1024)
print(f"Text size in memory: {size_mb:.2f} MB")

Text size in memory: 1.06 MB


In [6]:
strng_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_strng = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [strng_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_strng[i] for i in l])

# encoded_hello = torch.tensor(encode('hello'),dtype=torch.long)
# decoded_hello = decode(encoded_hello.tolist())

data = torch.tensor(encode(text), dtype=torch.long)

In [7]:
n = int(0.9*len(data))

train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split=='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #print(ix)
    x= torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    #return x,y
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

In [8]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [9]:
class Head(nn.Module):
    """ one head of self attention"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        #input size(Batch, time-step, channels)
        #output size(Batch, time-step, headsize)
        B,T,C = x.shape
        k = self.key(x) # (B,T,head_s)
        q = self.query(x) #(B,T,head_s)
        #compute attention scores, ('affinities')
        wei = q @ k.transpose(-2,-1) * k.shape[-1]** -0.5 #(B,T,head_s) @ (B, head_s,T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] ==0,float('-inf')) #(B, T, T)
        wei = F.softmax(wei, dim=-1) #(B, T, T)
        wei = self.dropout(wei)
        #perform the weighted aggregation of the values
        v = self.value(x)  # (B,T,head_s)
        out = wei @ v #(B,T,T) @ (B,T,head_s) -> (B,T,head_s)
        return out


class MultiHeadAttention(nn.Module):
    """ multiple heads of self attention in parallel """
    
    def __init__(self, n_head, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_head)])
        self.proj = nn.Linear(head_size * n_head, n_embed) # bringing back n_embed from: head_size = n_embed // n_head
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # (B, T, C(feautures like in head)
        out = self.dropout(self.proj(out))
        return out
        

class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed), # 4 is hyperparameter, Expanding dimentional for learning..hidden dim of FFN is 4 times (sweetspot)
            nn.ReLU(),
            nn.Linear(4* n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self,x):
        return self.net(x)
        

class Block(nn.Module):
    """ Transformer block: communication followed by computation"""

    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head #num of features(dimention)  captured in each MHA
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed) # postnorm
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self,x):
        """ here y is not target, after computation on x so later x can be added as resnet"""
        #post-norm like attention paper
        y = self.sa(x)
        x = self.ln1(x+y) # the Residual part here just added the previuos x ..y(x) + x(resnet)
        y = self.ffwd(x)
        x = self.ln2(x+y)
        return x

In [10]:
class GPTLanguagemodel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)# vocab_size)
        self.positional_embedding_table = nn.Embedding(block_size, n_embed)# vocab_size)
        self.blocks = nn.Sequential(*[Block(n_embed, n_head=n_head) for _ in range(n_layer)])

        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        
    def forward(self, index, targets=None):
        B,T = index.shape
        assert T <= block_size, f"Cannot forward sequence of length {T}, block size is only {block_size}"
        
        tok_emb = self.token_embedding_table(index) # (B, T, C)
        pos_emb = self.positional_embedding_table(torch.arange(0,T, device=device))# (T, C)
        x = tok_emb + pos_emb #(B,T, C)
        x = self.blocks(x) #(B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        # ---- Compute loss only if targets are provided (training mode) ----
        if targets is None:
            loss = None
        else:
            # Flatten both tensors to feed into cross_entropy
            B,T,C = logits.shape  #T "time" is the sequence size or block_size, C "channel" is vocab_size
            logits = logits.view(B*T, C) # B*T act as total number of samples, C class scores
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) #cross_Entropy expects input:(N,C), targets:(N,)
            
        return logits,loss

    def generate(self, index, max_new_tokens):
        #index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get predictions
            #cropping the context ( idx) to the last block_size tokens, otherwise our posi_emb will run out of scope
            idx_cond = index[:, -block_size:]
            logits, loss = self.forward(idx_cond)
            #for generation (not training) targets is None so skips logits flattening -> logtis become (B,T,C)
            logits = logits[:,-1,:] #becomes (B, C), -1 gets only the last token from the time step
            #apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B,C), dim=-1 acts across C class channels
            #sample for distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            #append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1), concatenates across time sequence, if dim=0 it would stack batches
        return index


In [11]:

model = GPTLanguagemodel(vocab_size).to(device)
#add torch compile 
m = torch.compile(model)

wandb.watch(m)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


7.537217 M parameters


In [12]:

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scaler = GradScaler()

start_time = datetime.now()

# Profile only limited steps (avoid big overhead)
profile_steps = 3  # or 50 if you really need longer profiling

for iter in range(max_iters):
    # ---- Evaluation & WandB logging ----
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train {losses['train']:.4f}, val {losses['val']:.4f}")
        wandb.log({
            "step": iter,
            "train_loss": losses["train"],
            "val_loss": losses["val"]
        })

    xb, yb = get_batch("train")

    # ---- Profiling only for first few steps ----
    if iter < profile_steps:
        with profile(
            activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
            record_shapes=False,
            profile_memory=False,
            with_stack=False
        ) as prof:
            with record_function(f"train_step_{iter}"):
                optimizer.zero_grad(set_to_none=True)
                with autocast():
                    logits, loss = model(xb, yb)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        # ---- Concise summary ----
        events = prof.key_averages()
        total_cuda_time = sum(e.cuda_time_total for e in events)
        top_ops = sorted(events, key=lambda e: e.cuda_time_total, reverse=True)[:5]

        print(f"\n---- Profiler Summary (Step {iter}) ----")
        for op in top_ops:
            print(f"{op.key:<40} {op.cuda_time_total/1000:.2f} ms")
        print(f"Total CUDA time: {total_cuda_time/1000:.2f} ms")

        # ---- Log to wandb (short summary) ----
        wandb.log({
            f"profiler/step_{iter}_cuda_ms": total_cuda_time / 1000,
            f"profiler/top_ops_step_{iter}": wandb.Html(
                "<pre>" + "\n".join([f"{op.key}: {op.cuda_time_total/1000:.2f} ms" for op in top_ops]) + "</pre>"
            )
        })

        # Optional trace export for Chrome/TensorBoard
        # prof.export_chrome_trace(f"trace_step_{iter}.json")

    else:
        # ---- Regular training ----
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits, loss = model(xb, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    torch.cuda.empty_cache()

# ---- Training summary ----
print(f"\nFinal loss: {loss.item():.4f}")

end_time = datetime.now()
total_minutes = (end_time - start_time).total_seconds() / 60
print(f"\nTotal training time: {total_minutes:.2f} minutes")

wandb.log({"total_training_minutes": total_minutes})


step 0: train 4.1900, val 4.1936


STAGE:2025-10-11 08:37:38 60387:60387 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-10-11 08:37:38 60387:60387 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-10-11 08:37:38 60387:60387 ActivityProfilerController.cpp:322] Completed Stage: Post Processing



---- Profiler Summary (Step 0) ----
train_step_0                             135.68 ms
aten::linear                             74.62 ms
aten::copy_                              64.22 ms
aten::_to_copy                           52.00 ms
aten::matmul                             47.68 ms
Total CUDA time: 1359.45 ms


STAGE:2025-10-11 08:37:39 60387:60387 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-10-11 08:37:39 60387:60387 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-10-11 08:37:39 60387:60387 ActivityProfilerController.cpp:322] Completed Stage: Post Processing



---- Profiler Summary (Step 1) ----
train_step_1                             99.85 ms
aten::copy_                              60.63 ms
aten::_to_copy                           47.05 ms
autograd::engine::evaluate_function: ToCopyBackward0 46.38 ms
aten::to                                 43.72 ms
Total CUDA time: 1184.60 ms


STAGE:2025-10-11 08:37:40 60387:60387 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-10-11 08:37:40 60387:60387 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-10-11 08:37:40 60387:60387 ActivityProfilerController.cpp:322] Completed Stage: Post Processing



---- Profiler Summary (Step 2) ----
train_step_2                             99.93 ms
aten::copy_                              60.49 ms
aten::_to_copy                           46.95 ms
autograd::engine::evaluate_function: ToCopyBackward0 46.20 ms
aten::to                                 41.45 ms
Total CUDA time: 1175.72 ms
step 200: train 2.4531, val 2.4746
step 400: train 2.4001, val 2.4428
step 600: train 2.2806, val 2.3550
step 800: train 2.0483, val 2.1658


/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/wandb/wandb_torch.py:175: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /opt/conda/conda-bld/pytorch_1699449201336/work/torch/csrc/tensor/python_tensor.cpp:83.)
  check = torch.cuda.FloatTensor(1).fill_(0)


step 1000: train 1.8372, val 1.9777

Final loss: 1.9688

Total training time: 11.21 minutes


In [ ]:
#with pytorch profiler and mixed precision
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = datetime.now()

scaler = GradScaler()

# Profile only a small number of steps to avoid overhead
profile_steps = 50

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")
        wandb.log({"steps": iter, "train_loss": losses["train"], "val_loss": losses["val"]})

    xb, yb = get_batch("train")

    # 🔹 Run profiler only for the first few steps
    if iter < profile_steps:
        with profile(
            activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
            record_shapes=True,
            profile_memory=True,
            with_stack=False
        ) as prof:
            with record_function(f"train_step_{iter}"):
                optimizer.zero_grad(set_to_none=True)
                with autocast():
                    logits, loss = model(xb, yb)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
    else:
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits, loss = model(xb, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    # Free unused cached memory
    torch.cuda.empty_cache()

    # 🔹 After profiling step, print and log results
    if iter < profile_steps:
        print(f"\n---- Profiler Report (Step {iter}) ----")
        report = prof.key_averages().table(sort_by="cuda_time_total", row_limit=10)
        print(report)

        # Log profiler summary to wandb as text
        wandb.log({f"profiler_step_{iter}": wandb.Html(f"<pre>{report}</pre>")})

        # Optionally export for Chrome or TensorBoard
        #prof.export_chrome_trace(f"trace_step_{iter}.json")

print(f"\nFinal loss: {loss.item():.4f}")

# 🔹 Time summary
end_time = datetime.now()
total_time = end_time - start_time
total_minutes = total_time.total_seconds() / 60
print(f"\nTotal training time: {total_minutes:.2f} minutes")
wandb.log({"total_training_minutes": total_minutes})


step: 0, train loss: 2.4964, val loss: 2.5069


STAGE:2025-10-11 08:23:18 49067:49067 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-10-11 08:23:18 49067:49067 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-10-11 08:23:18 49067:49067 ActivityProfilerController.cpp:322] Completed Stage: Post Processing



---- Profiler Report (Step 0) ----
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           train_step_0        11.24%      32.193ms        92.90%     265.964ms     265.964ms       0.000us         0.00%      93.824ms      93.824ms         376 b     

STAGE:2025-10-11 08:23:19 49067:49067 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-10-11 08:23:20 49067:49067 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-10-11 08:23:20 49067:49067 ActivityProfilerController.cpp:322] Completed Stage: Post Processing



---- Profiler Report (Step 1) ----
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           train_step_1        13.19%      37.968ms        90.31%     260.030ms     260.030ms       0.000us         0.00%      93.711ms      93.711ms           0 b     

STAGE:2025-10-11 08:23:21 49067:49067 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-10-11 08:23:21 49067:49067 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-10-11 08:23:21 49067:49067 ActivityProfilerController.cpp:322] Completed Stage: Post Processing



---- Profiler Report (Step 2) ----
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           train_step_2        13.70%      38.615ms        92.56%     260.943ms     260.943ms       0.000us         0.00%      93.865ms      93.865ms           0 b     

STAGE:2025-10-11 08:23:22 49067:49067 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-10-11 08:23:22 49067:49067 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-10-11 08:23:22 49067:49067 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = datetime.now()
for iter in range(max_iters):
    if iter % eval_interval  ==0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}")
        
        #wandb log
        wandb.log({"steps": iter,"train_loss": losses["train"], "val_loss": losses["val"]})

    #sample a batch of data
    xb,yb = get_batch("train")

    #evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)#make grad star overnot accumulated
    loss.backward()
    optimizer.step()

    #free up unoccupied cached memory
    torch.cuda.empty_cache()
    
print(loss.item())
end_time = datetime.now()
total_time = end_time - start_time
total_minutes = total_time.total_seconds() / 60
print("total time spend in training: {total_minutes:.4f} minutes")

step: 0, train loss: 4.3889, val loss: 4.3935
step: 100, train loss: 2.4802, val loss: 2.5027
step: 200, train loss: 2.4508, val loss: 2.4868
2.4638798236846924
total time spend in training:, {total_minutes:.4f} minutes


In [14]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


Vis kee, tere, lal havin, and,
Anou, t we woro othingatsttird Ieame! f tin areasppo w wikemod anger on cld, teno me easot n
When thand Anghand lon camorl folarearsen? benwith alavee y w s oveleay anthisco s o l;
Bughtese hem s hadat'vey tlle mer Shemerofour meecou anarb
th kit h hen anareh's hintord it wind laineaaye ve thathtoseldesheruke thak'en we hable hoseamim fuertrut tha cochaiese; tlt wofapestiroffou;
ICUnouty hin,
LEYood dy,
t my, arses t pe th mande
Fay oth malyskst s,

ARET:
DUCELI ch
